# Pre-Training a Mini GPT (124M Parameters) Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Embedding Layer

Token embeddings map each of the 50,257 possible tokens to a 768-dimensional vector. Position embeddings add information about where each token sits in the sequence. The two are summed.

In [ ]:
```python

import numpy as np

class Embedding:

    def __init__(self, vocab_size, embed_dim, max_seq_len):

        self.token_embed = np.random.randn(vocab_size, embed_dim) * 0.02

        self.pos_embed = np.random.randn(max_seq_len, embed_dim) * 0.02

    def forward(self, token_ids):

        seq_len = token_ids.shape[-1]

        tok_emb = self.token_embed[token_ids]

        pos_emb = self.pos_embed[:seq_len]

        return tok_emb + pos_emb

In [ ]:
```

The 0.02 standard deviation for initialization comes from the GPT-2 paper. Too large and the initial forward passes produce extreme values that destabilize training. Too small and the initial outputs are nearly identical for all inputs, making early gradient signals useless.

### Step 2: Self-Attention with Causal Mask

Single-head attention first. The causal mask sets future positions to negative infinity before softmax, ensuring each position can only attend to itself and earlier positions.

In [ ]:
```python

def attention(Q, K, V, mask=None):

    d_k = Q.shape[-1]

    scores = Q @ K.transpose(0, -1, -2 if Q.ndim == 4 else 1) / np.sqrt(d_k)

    if mask is not None:

        scores = scores + mask

    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))

    weights = weights / weights.sum(axis=-1, keepdims=True)

    return weights @ V

In [ ]:
```

The softmax implementation subtracts the maximum before exponentiating. Without this, exp(large_number) overflows to infinity. This is a numerical stability trick that does not change the output because softmax(x - c) = softmax(x) for any constant c.

### Step 3: Multi-Head Attention

Split the 768-dimensional input into 12 heads of 64 dimensions each. Each head computes attention independently. Concatenate the results and project back to 768 dimensions.

In [ ]:
```python

class MultiHeadAttention:

    def __init__(self, embed_dim, num_heads):

        self.num_heads = num_heads

        self.head_dim = embed_dim // num_heads

        self.W_q = np.random.randn(embed_dim, embed_dim) * 0.02

        self.W_k = np.random.randn(embed_dim, embed_dim) * 0.02

        self.W_v = np.random.randn(embed_dim, embed_dim) * 0.02

        self.W_out = np.random.randn(embed_dim, embed_dim) * 0.02

    def forward(self, x, mask=None):

        batch, seq_len, d = x.shape

        Q = (x @ self.W_q).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        K = (x @ self.W_k).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        V = (x @ self.W_v).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(self.head_dim)

        if mask is not None:

            scores = scores + mask

        weights = np.exp(scores - scores.max(axis=-1, keepdims=True))

        weights = weights / weights.sum(axis=-1, keepdims=True)

        attn_out = weights @ V

        attn_out = attn_out.transpose(0, 2, 1, 3).reshape(batch, seq_len, d)

        return attn_out @ self.W_out

In [ ]:
```

The reshape-transpose-reshape dance is the most confusing part of multi-head attention. Here is what happens: the (batch, seq_len, 768) tensor becomes (batch, seq_len, 12, 64), then (batch, 12, seq_len, 64). Now each of the 12 heads has its own (seq_len, 64) matrix to run attention on. After attention, we reverse the process: (batch, 12, seq_len, 64) becomes (batch, seq_len, 12, 64) becomes (batch, seq_len, 768).

### Step 4: Transformer Block

One complete transformer block: LayerNorm, multi-head attention with residual, LayerNorm, feedforward with residual.

In [ ]:
```python

class LayerNorm:

    def __init__(self, dim, eps=1e-5):

        self.gamma = np.ones(dim)

        self.beta = np.zeros(dim)

        self.eps = eps

    def forward(self, x):

        mean = x.mean(axis=-1, keepdims=True)

        var = x.var(axis=-1, keepdims=True)

        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta

class FeedForward:

    def __init__(self, embed_dim, ff_dim):

        self.W1 = np.random.randn(embed_dim, ff_dim) * 0.02

        self.b1 = np.zeros(ff_dim)

        self.W2 = np.random.randn(ff_dim, embed_dim) * 0.02

        self.b2 = np.zeros(embed_dim)

    def forward(self, x):

        h = x @ self.W1 + self.b1

        h = np.maximum(0, h)  # GELU approximation: ReLU for simplicity

        return h @ self.W2 + self.b2

class TransformerBlock:

    def __init__(self, embed_dim, num_heads, ff_dim):

        self.ln1 = LayerNorm(embed_dim)

        self.attn = MultiHeadAttention(embed_dim, num_heads)

        self.ln2 = LayerNorm(embed_dim)

        self.ffn = FeedForward(embed_dim, ff_dim)

    def forward(self, x, mask=None):

        x = x + self.attn.forward(self.ln1.forward(x), mask)

        x = x + self.ffn.forward(self.ln2.forward(x))

        return x

In [ ]:
```

The feedforward network expands the 768-dimensional input to 3,072 dimensions (4x), applies a nonlinearity, then projects back to 768. This expansion-contraction pattern gives the model a "wider" internal representation to work with at each position. GPT-2 uses GELU activation, but we use ReLU here for simplicity -- the difference is minor for understanding the architecture.

### Step 5: Full GPT Model

Stack 12 transformer blocks. Add the embedding layer at the front and the output projection at the back.

In [ ]:
```python

class MiniGPT:

    def __init__(self, vocab_size=50257, embed_dim=768, num_heads=12,

                 num_layers=12, max_seq_len=1024, ff_dim=3072):

        self.embedding = Embedding(vocab_size, embed_dim, max_seq_len)

        self.blocks = [

            TransformerBlock(embed_dim, num_heads, ff_dim)

            for _ in range(num_layers)

        ]

        self.ln_f = LayerNorm(embed_dim)

        self.vocab_size = vocab_size

        self.embed_dim = embed_dim

    def forward(self, token_ids):

        seq_len = token_ids.shape[-1]

        mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)

        x = self.embedding.forward(token_ids)

        for block in self.blocks:

            x = block.forward(x, mask)

        x = self.ln_f.forward(x)

        logits = x @ self.embedding.token_embed.T

        return logits

    def count_parameters(self):

        total = 0

        total += self.embedding.token_embed.size

        total += self.embedding.pos_embed.size

        for block in self.blocks:

            total += block.attn.W_q.size + block.attn.W_k.size

            total += block.attn.W_v.size + block.attn.W_out.size

            total += block.ffn.W1.size + block.ffn.b1.size

            total += block.ffn.W2.size + block.ffn.b2.size

            total += block.ln1.gamma.size + block.ln1.beta.size

            total += block.ln2.gamma.size + block.ln2.beta.size

        total += self.ln_f.gamma.size + self.ln_f.beta.size

        return total

In [ ]:
```

Notice the weight tying: `logits = x @ self.embedding.token_embed.T`. The output projection reuses the token embedding matrix (transposed). This is not just a parameter-saving trick. It means the model uses the same vector space for understanding tokens (embeddings) and predicting them (output).

### Step 6: Training Loop

For a real training run on 124M parameters, you would need a GPU and PyTorch. This training loop demonstrates the mechanics on a small model that runs in pure numpy. We use a tiny model (4 layers, 4 heads, 128 dims) to make it tractable.

In [ ]:
```python

def cross_entropy_loss(logits, targets):

    batch, seq_len, vocab_size = logits.shape

    logits_flat = logits.reshape(-1, vocab_size)

    targets_flat = targets.reshape(-1)

    max_logits = logits_flat.max(axis=-1, keepdims=True)

    log_softmax = logits_flat - max_logits - np.log(

        np.exp(logits_flat - max_logits).sum(axis=-1, keepdims=True)

    )

    loss = -log_softmax[np.arange(len(targets_flat)), targets_flat].mean()

    return loss

def train_mini_gpt(text, vocab_size=256, embed_dim=128, num_heads=4,

                   num_layers=4, seq_len=64, num_steps=200, lr=3e-4):

    tokens = np.array(list(text.encode("utf-8")[:2048]))

    model = MiniGPT(

        vocab_size=vocab_size, embed_dim=embed_dim, num_heads=num_heads,

        num_layers=num_layers, max_seq_len=seq_len, ff_dim=embed_dim * 4

    )

    print(f"Model parameters: {model.count_parameters():,}")

    print(f"Training tokens: {len(tokens):,}")

    print(f"Config: {num_layers} layers, {num_heads} heads, {embed_dim} dims")

    print()

    for step in range(num_steps):

        start_idx = np.random.randint(0, max(1, len(tokens) - seq_len - 1))

        batch_tokens = tokens[start_idx:start_idx + seq_len + 1]

        input_ids = batch_tokens[:-1].reshape(1, -1)

        target_ids = batch_tokens[1:].reshape(1, -1)

        logits = model.forward(input_ids)

        loss = cross_entropy_loss(logits, target_ids)

        if step % 20 == 0:

            print(f"Step {step:4d} | Loss: {loss:.4f}")

    return model

In [ ]:
```

The loss starts near ln(vocab_size) -- for a 256-token byte-level vocabulary, that is ln(256) = 5.55. A random model assigns equal probability to every token. As training progresses, the loss drops because the model learns to predict common patterns: "th" after "t", space after a period, and so on.

In production, you would use Adam optimizer with gradient accumulation, learning rate warmup, and gradient clipping. The forward-pass-loss-backward-update loop is identical. The optimizer is more sophisticated.

### Step 7: Text Generation

Generation uses the trained model to predict one token at a time. Each prediction is sampled from the output distribution (or taken greedily as the argmax).

In [ ]:
```python

def generate(model, prompt_tokens, max_new_tokens=100, temperature=0.8):

    tokens = list(prompt_tokens)

    seq_len = model.embedding.pos_embed.shape[0]

    for _ in range(max_new_tokens):

        context = np.array(tokens[-seq_len:]).reshape(1, -1)

        logits = model.forward(context)

        next_logits = logits[0, -1, :]

        next_logits = next_logits / temperature

        probs = np.exp(next_logits - next_logits.max())

        probs = probs / probs.sum()

        next_token = np.random.choice(len(probs), p=probs)

        tokens.append(next_token)

    return tokens

In [ ]:
```

Temperature controls randomness. Temperature 1.0 uses the raw distribution. Temperature 0.5 sharpens it (more deterministic -- the model picks its top choices more often). Temperature 1.5 flattens it (more random -- low-probability tokens get a bigger chance). Temperature 0.0 is greedy decoding (always pick the highest probability token).

The `tokens[-seq_len:]` window is necessary because the model has a maximum context length (1024 for GPT-2). Once you exceed it, you must drop the oldest tokens. This is the "context window" that everyone talks about.

## Exercises

In [ ]:
1. Modify the model to use 24 layers and 16 heads instead of 12/12. Count the parameters. How does doubling the depth compare to doubling the width (embedding dimension)?

2. Implement the GELU activation function (GELU(x) = x * 0.5 * (1 + erf(x / sqrt(2)))) and replace the ReLU in the feedforward network. Run training for 500 steps with each activation and compare the final loss.

3. Add a KV cache to the generation function. Store K and V tensors for each layer after the first forward pass, and reuse them for subsequent tokens. Measure the speedup: generate 200 tokens with and without the cache and compare wall-clock time.

4. Implement top-k sampling (only consider the k highest-probability tokens) and top-p sampling (nucleus sampling: consider the smallest set of tokens whose cumulative probability exceeds p). Compare the output quality at temperature 0.8 with top-k=50 vs top-p=0.95.

5. Build a training loss curve plotter. Train the model for 1000 steps and plot loss vs step. Identify the three phases: rapid initial descent (learning common bytes), slower middle phase (learning byte patterns), and plateau (overfitting on the small corpus). The shape of this curve is the same whether you are training a 128-dim model or GPT-4.